In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

dataset = pd.read_csv('data/train.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\W+', ' ', text)

    return text

dataset['text'] = dataset['text'].apply(clean_text)
text = dataset["text"].tolist()
target = dataset["target"].tolist()
x_train, x_eval, y_train, y_eval = train_test_split(text, target, test_size=0.2)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf_vect = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf_vect.fit_transform(x_train)
y_train = np.array(y_train)

X_train_tfidf.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression().fit(X_train_tfidf, y_train)

X_eval_tfidf = tfidf_vect.transform(x_eval)
y_eval = np.array(y_eval)
eval_predicted = clf.predict(X_eval_tfidf)

from sklearn import metrics

print(metrics.classification_report(y_eval, eval_predicted))
metrics.confusion_matrix(y_eval, eval_predicted)

In [ ]:
test = pd.read_csv('data/test.csv')

test['text'] = test['text'].apply(clean_text)
test_text = test["text"].tolist()

X_test_tfidf = tfidf_vect.transform(test_text)
test_predicted = clf.predict(X_test_tfidf)

df = pd.DataFrame(test_predicted, columns=['target'])
submission_logic = pd.DataFrame({'id': test['id'], 'target': test_predicted})

submission_logic.to_csv('outputs/submission_logistic.csv', index=False)
print("Submission saved!")

## Optimization solution: Use LinearSVC and improved TF-IDF

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# Establish a pipeline that integrates feature extraction and classifiers.
# Using sublinear_tf=True can smooth out the influence of extremely high-frequency words.
# LinearSVC typically performs better when handling high-dimensional sparse features (such as TF-IDF).
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2),
                             max_features=50000,
                             sublinear_tf=True)),
    ('clf', LinearSVC(C=0.5, random_state=42))
])

text_clf.fit(x_train, y_train)

optimized_predicted = text_clf.predict(x_eval)
print("Optimized classification report：")
print(metrics.classification_report(y_eval, optimized_predicted))
print("Confusion Matrix：")
display(metrics.confusion_matrix(y_eval, optimized_predicted))

In [ ]:
optimized_test_predicted = text_clf.predict(test_text)
submission_optimized = pd.DataFrame({'id': test['id'], 'target': optimized_test_predicted})

output_path = 'outputs/submission_linearSVC.csv'
submission_optimized.to_csv(output_path, index=False)
print(f"Submission_linearSVC saved!")